In [ ]:
import os
import gc
import random
import warnings
import numpy as np
import pandas as pd

# ===== 一定要放在 import tensorflow 前 =====
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"   # 強制 CPU

import tensorflow as tf

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, average_precision_score
)
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, BatchNormalization,
    LayerNormalization, Layer, GRU, Reshape
)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2


# =========================================================
# A. 基本設定
# =========================================================
warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# CPU 執行緒可視電腦調整
tf.config.threading.set_intra_op_parallelism_threads(8)
tf.config.threading.set_inter_op_parallelism_threads(2)


# =========================================================
# B. 讀資料
# =========================================================
DATA_PATH = "final_dataset_for_ml_FULL.csv"
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")


# =========================================================
# C. 反轉反向指標
# =========================================================
reverse_map = {
    "llama_vagueness_score_1": "llama_vagueness_risk_1",
    "llama_deflection_score_1": "llama_deflection_risk_1"
}

for raw_col, risk_col in reverse_map.items():
    if raw_col not in df.columns:
        raise ValueError(f"{raw_col} 不存在於資料中")
    df[risk_col] = 1 - df[raw_col].clip(0, 1)


# =========================================================
# D. target / groups
# =========================================================
if "label" not in df.columns:
    raise ValueError("label 不存在於資料中")
if "Company" not in df.columns:
    raise ValueError("Company 不存在於資料中")

y = df["label"].astype(int)
groups_all = df["Company"]


# =========================================================
# E. 特徵欄位
# =========================================================
semantic_cols = [
    "llama_specificity_score_1",
    "llama_evidence_substantiation_score_1",
    "llama_vagueness_risk_1",
    "llama_commitment_score_1",
    "llama_temporal_credibility_score_1",
    "llama_deflection_risk_1",
    "llama_comparability_score_1"
]

lexical_cols = [
    "llama_has_scope_1",
    "llama_has_sbti_1",
    "llama_has_material_1",
    "llama_has_kpi_1",
    "llama_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

all_needed_cols = semantic_cols + lexical_cols + financial_cols + ["label", "Company"]
missing_cols = [c for c in all_needed_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")

# ✅ 改成真正 M1~M6
feature_sets = {
    "M1: Semantic": semantic_cols,
    "M2: Lexical": lexical_cols,
    "M3: Financial": financial_cols,
    "M4: Semantic + Lexical": semantic_cols + lexical_cols,
    "M5: Semantic + Financial": semantic_cols + financial_cols,
    "M6: Semantic + Lexical + Financial": semantic_cols + lexical_cols + financial_cols
}


# =========================================================
# F. CV 設定（瘦身）
# =========================================================
outer_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)


# =========================================================
# G. Attention Layer
# =========================================================
class TemporalAttention(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        hidden_dim = input_shape[-1]
        self.W = self.add_weight(
            name="att_weight",
            shape=(hidden_dim, hidden_dim),
            initializer="glorot_uniform",
            trainable=True
        )
        self.b = self.add_weight(
            name="att_bias",
            shape=(hidden_dim,),
            initializer="zeros",
            trainable=True
        )
        self.u = self.add_weight(
            name="att_u",
            shape=(hidden_dim, 1),
            initializer="glorot_uniform",
            trainable=True
        )
        super().build(input_shape)

    def call(self, inputs):
        score = tf.tanh(tf.tensordot(inputs, self.W, axes=1) + self.b)
        attn = tf.tensordot(score, self.u, axes=1)
        attn = tf.nn.softmax(attn, axis=1)
        context = tf.reduce_sum(inputs * attn, axis=1)
        return context


# =========================================================
# H. 建模型（瘦身版）
# =========================================================
def build_rnn_attention_light(
    num_features,
    rnn_units=8,
    dropout_rate=0.2,
    learning_rate=1e-3,
    dense_units=8,
    l2_reg=1e-4
):
    inputs = Input(shape=(num_features,), dtype=tf.float32, name="feature_input")

    # 直接 reshape，不做 gating
    x = Reshape((num_features, 1), name="reshape_to_seq")(inputs)

    x = GRU(
        rnn_units,
        return_sequences=True,
        dropout=dropout_rate,
        recurrent_dropout=0.0,
        kernel_regularizer=l2(l2_reg),
        name="gru"
    )(x)

    x = TemporalAttention(name="temporal_attention")(x)
    x = LayerNormalization(name="ln_after_att")(x)

    x = Dense(
        dense_units,
        activation="relu",
        kernel_regularizer=l2(l2_reg),
        name="dense_1"
    )(x)
    x = BatchNormalization(name="bn_1")(x)
    x = Dropout(dropout_rate, name="dropout_1")(x)

    outputs = Dense(1, activation="sigmoid", name="output")(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


# =========================================================
# I. feature split
# =========================================================
def split_feature_columns(selected_cols):
    cont_cols = [c for c in selected_cols if c in (semantic_cols + financial_cols)]
    lex_cols = [c for c in selected_cols if c in lexical_cols]
    return cont_cols, lex_cols


# =========================================================
# J. 前處理
# =========================================================
def preprocess_train_test(X_tr_df, X_te_df, selected_cols):
    cont_cols, lex_cols = split_feature_columns(selected_cols)

    if len(cont_cols) > 0:
        cont_imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()

        X_tr_cont = cont_imputer.fit_transform(X_tr_df[cont_cols])
        X_te_cont = cont_imputer.transform(X_te_df[cont_cols])

        X_tr_cont = scaler.fit_transform(X_tr_cont)
        X_te_cont = scaler.transform(X_te_cont)
    else:
        X_tr_cont = np.zeros((len(X_tr_df), 0), dtype=np.float32)
        X_te_cont = np.zeros((len(X_te_df), 0), dtype=np.float32)

    if len(lex_cols) > 0:
        lex_imputer = SimpleImputer(strategy="most_frequent")

        X_tr_lex = lex_imputer.fit_transform(X_tr_df[lex_cols]).astype(np.float32)
        X_te_lex = lex_imputer.transform(X_te_df[lex_cols]).astype(np.float32)
    else:
        X_tr_lex = np.zeros((len(X_tr_df), 0), dtype=np.float32)
        X_te_lex = np.zeros((len(X_te_df), 0), dtype=np.float32)

    X_tr = np.concatenate([X_tr_cont, X_tr_lex], axis=1).astype(np.float32)
    X_te = np.concatenate([X_te_cont, X_te_lex], axis=1).astype(np.float32)

    return X_tr, X_te


# =========================================================
# K. 快速推論
# =========================================================
def fast_predict(model, X):
    X = np.asarray(X, dtype=np.float32)
    return model(X, training=False).numpy().ravel()


# =========================================================
# L. 清記憶體
# =========================================================
def cleanup_tf():
    tf.keras.backend.clear_session()
    gc.collect()


# =========================================================
# M. 評估單一 feature set
# =========================================================
def evaluate_feature_set(df, y, groups, feature_name, selected_cols):
    fold_metrics = []

    print("\n" + "=" * 80)
    print(f"Feature Set: {feature_name}")
    print("=" * 80)

    for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(df[selected_cols], y, groups=groups), start=1
    ):
        print(f"\n[{feature_name}] Fold {fold_idx}")

        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()

        X_train, X_test = preprocess_train_test(
            train_df[selected_cols],
            test_df[selected_cols],
            selected_cols
        )

        # 再從 train 切一小塊當 validation
        val_ratio = 0.15
        n_train = len(X_train)
        val_size = max(1, int(n_train * val_ratio))

        X_tr = X_train[:-val_size]
        X_val = X_train[-val_size:]
        y_tr = y_train.iloc[:-val_size]
        y_val = y_train.iloc[-val_size:]

        classes = np.unique(y_tr)
        class_weights = compute_class_weight(
            class_weight="balanced",
            classes=classes,
            y=y_tr
        )
        class_weight_dict = {int(cls): float(w) for cls, w in zip(classes, class_weights)}

        cleanup_tf()

        model = build_rnn_attention_light(
            num_features=X_tr.shape[1],
            rnn_units=8,
            dropout_rate=0.2,
            learning_rate=1e-3,
            dense_units=8
        )

        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=3,
            min_delta=1e-3,
            restore_best_weights=True
        )

        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=25,
            batch_size=32,
            class_weight=class_weight_dict,
            callbacks=[early_stop],
            verbose=0
        )

        test_prob = fast_predict(model, X_test)
        y_pred = (test_prob >= 0.5).astype(int)   # 固定 threshold 0.5

        try:
            roc_auc = roc_auc_score(y_test, test_prob)
        except ValueError:
            roc_auc = np.nan

        fold_result = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc,
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_test, test_prob)
        }
        fold_metrics.append(fold_result)

        del model
        cleanup_tf()

        print(
            f"[{feature_name}] Fold {fold_idx} done | "
            f"ACC={fold_result['accuracy']:.4f} | "
            f"F1={fold_result['f1']:.4f} | "
            f"P={fold_result['precision']:.4f} | "
            f"R={fold_result['recall']:.4f}"
        )

    row = {
        "Model": "Light RNN + Attention",
        "Feature_Set": feature_name,
        "Config": "{rnn_units: 8, dropout: 0.2, lr: 1e-3, dense: 8, threshold: 0.5}",
        "Num_Features": len(selected_cols),
        "Num_Continuous": len([c for c in selected_cols if c in (semantic_cols + financial_cols)]),
        "Num_Lexical": len([c for c in selected_cols if c in lexical_cols]),
        "Accuracy_mean": np.nanmean([m["accuracy"] for m in fold_metrics]),
        "F1_mean": np.nanmean([m["f1"] for m in fold_metrics]),
        "ROC_AUC_mean": np.nanmean([m["roc_auc"] for m in fold_metrics]),
        "Precision_mean": np.nanmean([m["precision"] for m in fold_metrics]),
        "Recall_mean": np.nanmean([m["recall"] for m in fold_metrics]),
        "PR_AUC_mean": np.nanmean([m["average_precision"] for m in fold_metrics]),
        "Accuracy_std": np.nanstd([m["accuracy"] for m in fold_metrics], ddof=1),
        "F1_std": np.nanstd([m["f1"] for m in fold_metrics], ddof=1),
        "ROC_AUC_std": np.nanstd([m["roc_auc"] for m in fold_metrics], ddof=1),
        "Precision_std": np.nanstd([m["precision"] for m in fold_metrics], ddof=1),
        "Recall_std": np.nanstd([m["recall"] for m in fold_metrics], ddof=1),
        "PR_AUC_std": np.nanstd([m["average_precision"] for m in fold_metrics], ddof=1),
    }
    return row


# =========================================================
# N. 執行 M1~M6
# =========================================================
all_results = []

for feature_name, cols in feature_sets.items():
    result_row = evaluate_feature_set(
        df=df,
        y=y,
        groups=groups_all,
        feature_name=feature_name,
        selected_cols=cols
    )
    all_results.append(result_row)


# =========================================================
# O. 結果整理
# =========================================================
results_df = pd.DataFrame(all_results)
numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

print("\n===== Final Results =====")
print(results_df)

output_name = "llama_light_rnn_attention_M1_M6_results.csv"
results_df.to_csv(output_name, index=False, encoding="utf-8-sig")

print(f"\nResults saved to: {output_name}")

I0000 00:00:1774929156.934393 1139536 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774929156.935697 1139536 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774929158.232577 1139536 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774929158.233544 1139536 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.



Feature Set: M1: Semantic

[M1: Semantic] Fold 1
[M1: Semantic] Fold 1 done | ACC=0.7273 | F1=0.5946 | P=0.4400 | R=0.9167

[M1: Semantic] Fold 2
[M1: Semantic] Fold 2 done | ACC=0.8246 | F1=0.5455 | P=0.3750 | R=1.0000

[M1: Semantic] Fold 3
[M1: Semantic] Fold 3 done | ACC=0.9487 | F1=0.5000 | P=0.5000 | R=0.5000

[M1: Semantic] Fold 4
[M1: Semantic] Fold 4 done | ACC=0.9726 | F1=0.0000 | P=0.0000 | R=0.0000

[M1: Semantic] Fold 5
[M1: Semantic] Fold 5 done | ACC=0.5692 | F1=0.3333 | P=0.2059 | R=0.8750

Feature Set: M2: Lexical

[M2: Lexical] Fold 1
[M2: Lexical] Fold 1 done | ACC=0.2182 | F1=0.3582 | P=0.2182 | R=1.0000

[M2: Lexical] Fold 2
[M2: Lexical] Fold 2 done | ACC=0.9474 | F1=0.8000 | P=0.6667 | R=1.0000

[M2: Lexical] Fold 3
[M2: Lexical] Fold 3 done | ACC=0.9487 | F1=0.0000 | P=0.0000 | R=0.0000

[M2: Lexical] Fold 4
[M2: Lexical] Fold 4 done | ACC=0.8493 | F1=0.0000 | P=0.0000 | R=0.0000

[M2: Lexical] Fold 5
[M2: Lexical] Fold 5 done | ACC=0.7538 | F1=0.3333 | P=0.250

In [ ]:
import os
import gc
import random
import warnings
import numpy as np
import pandas as pd

# ===== 一定要放在 import tensorflow 前 =====
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"   # 強制 CPU

import tensorflow as tf

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, average_precision_score
)
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, BatchNormalization,
    LayerNormalization, Layer, GRU, Reshape
)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2


# =========================================================
# A. 基本設定
# =========================================================
warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# CPU 執行緒可視電腦調整
tf.config.threading.set_intra_op_parallelism_threads(8)
tf.config.threading.set_inter_op_parallelism_threads(2)


# =========================================================
# B. 讀資料
# =========================================================
DATA_PATH = "final_dataset_for_ml_FULL.csv"
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")


# =========================================================
# C. 反轉反向指標
# =========================================================
reverse_map = {
    "chatgpt_vagueness_score_1": "chatgpt_vagueness_risk_1",
    "chatgpt_deflection_score_1": "chatgpt_deflection_risk_1"
}

for raw_col, risk_col in reverse_map.items():
    if raw_col not in df.columns:
        raise ValueError(f"{raw_col} 不存在於資料中")
    df[risk_col] = 1 - df[raw_col].clip(0, 1)


# =========================================================
# D. target / groups
# =========================================================
if "label" not in df.columns:
    raise ValueError("label 不存在於資料中")
if "Company" not in df.columns:
    raise ValueError("Company 不存在於資料中")

y = df["label"].astype(int)
groups_all = df["Company"]


# =========================================================
# E. 特徵欄位
# =========================================================
semantic_cols = [
    "chatgpt_specificity_score_1",
    "chatgpt_evidence_substantiation_score_1",
    "chatgpt_vagueness_risk_1",
    "chatgpt_commitment_score_1",
    "chatgpt_temporal_credibility_score_1",
    "chatgpt_deflection_risk_1",
    "chatgpt_comparability_score_1"
]

lexical_cols = [
    "chatgpt_has_scope_1",
    "chatgpt_has_sbti_1",
    "chatgpt_has_material_1",
    "chatgpt_has_kpi_1",
    "chatgpt_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

all_needed_cols = semantic_cols + lexical_cols + financial_cols + ["label", "Company"]
missing_cols = [c for c in all_needed_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")

# ✅ 改成真正 M1~M6
feature_sets = {
    "M1: Semantic": semantic_cols,
    "M2: Lexical": lexical_cols,
    "M3: Financial": financial_cols,
    "M4: Semantic + Lexical": semantic_cols + lexical_cols,
    "M5: Semantic + Financial": semantic_cols + financial_cols,
    "M6: Semantic + Lexical + Financial": semantic_cols + lexical_cols + financial_cols
}


# =========================================================
# F. CV 設定（瘦身）
# =========================================================
outer_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)


# =========================================================
# G. Attention Layer
# =========================================================
class TemporalAttention(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        hidden_dim = input_shape[-1]
        self.W = self.add_weight(
            name="att_weight",
            shape=(hidden_dim, hidden_dim),
            initializer="glorot_uniform",
            trainable=True
        )
        self.b = self.add_weight(
            name="att_bias",
            shape=(hidden_dim,),
            initializer="zeros",
            trainable=True
        )
        self.u = self.add_weight(
            name="att_u",
            shape=(hidden_dim, 1),
            initializer="glorot_uniform",
            trainable=True
        )
        super().build(input_shape)

    def call(self, inputs):
        score = tf.tanh(tf.tensordot(inputs, self.W, axes=1) + self.b)
        attn = tf.tensordot(score, self.u, axes=1)
        attn = tf.nn.softmax(attn, axis=1)
        context = tf.reduce_sum(inputs * attn, axis=1)
        return context


# =========================================================
# H. 建模型（瘦身版）
# =========================================================
def build_rnn_attention_light(
    num_features,
    rnn_units=8,
    dropout_rate=0.2,
    learning_rate=1e-3,
    dense_units=8,
    l2_reg=1e-4
):
    inputs = Input(shape=(num_features,), dtype=tf.float32, name="feature_input")

    # 直接 reshape，不做 gating
    x = Reshape((num_features, 1), name="reshape_to_seq")(inputs)

    x = GRU(
        rnn_units,
        return_sequences=True,
        dropout=dropout_rate,
        recurrent_dropout=0.0,
        kernel_regularizer=l2(l2_reg),
        name="gru"
    )(x)

    x = TemporalAttention(name="temporal_attention")(x)
    x = LayerNormalization(name="ln_after_att")(x)

    x = Dense(
        dense_units,
        activation="relu",
        kernel_regularizer=l2(l2_reg),
        name="dense_1"
    )(x)
    x = BatchNormalization(name="bn_1")(x)
    x = Dropout(dropout_rate, name="dropout_1")(x)

    outputs = Dense(1, activation="sigmoid", name="output")(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


# =========================================================
# I. feature split
# =========================================================
def split_feature_columns(selected_cols):
    cont_cols = [c for c in selected_cols if c in (semantic_cols + financial_cols)]
    lex_cols = [c for c in selected_cols if c in lexical_cols]
    return cont_cols, lex_cols


# =========================================================
# J. 前處理
# =========================================================
def preprocess_train_test(X_tr_df, X_te_df, selected_cols):
    cont_cols, lex_cols = split_feature_columns(selected_cols)

    if len(cont_cols) > 0:
        cont_imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()

        X_tr_cont = cont_imputer.fit_transform(X_tr_df[cont_cols])
        X_te_cont = cont_imputer.transform(X_te_df[cont_cols])

        X_tr_cont = scaler.fit_transform(X_tr_cont)
        X_te_cont = scaler.transform(X_te_cont)
    else:
        X_tr_cont = np.zeros((len(X_tr_df), 0), dtype=np.float32)
        X_te_cont = np.zeros((len(X_te_df), 0), dtype=np.float32)

    if len(lex_cols) > 0:
        lex_imputer = SimpleImputer(strategy="most_frequent")

        X_tr_lex = lex_imputer.fit_transform(X_tr_df[lex_cols]).astype(np.float32)
        X_te_lex = lex_imputer.transform(X_te_df[lex_cols]).astype(np.float32)
    else:
        X_tr_lex = np.zeros((len(X_tr_df), 0), dtype=np.float32)
        X_te_lex = np.zeros((len(X_te_df), 0), dtype=np.float32)

    X_tr = np.concatenate([X_tr_cont, X_tr_lex], axis=1).astype(np.float32)
    X_te = np.concatenate([X_te_cont, X_te_lex], axis=1).astype(np.float32)

    return X_tr, X_te


# =========================================================
# K. 快速推論
# =========================================================
def fast_predict(model, X):
    X = np.asarray(X, dtype=np.float32)
    return model(X, training=False).numpy().ravel()


# =========================================================
# L. 清記憶體
# =========================================================
def cleanup_tf():
    tf.keras.backend.clear_session()
    gc.collect()


# =========================================================
# M. 評估單一 feature set
# =========================================================
def evaluate_feature_set(df, y, groups, feature_name, selected_cols):
    fold_metrics = []

    print("\n" + "=" * 80)
    print(f"Feature Set: {feature_name}")
    print("=" * 80)

    for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(df[selected_cols], y, groups=groups), start=1
    ):
        print(f"\n[{feature_name}] Fold {fold_idx}")

        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()

        X_train, X_test = preprocess_train_test(
            train_df[selected_cols],
            test_df[selected_cols],
            selected_cols
        )

        # 再從 train 切一小塊當 validation
        val_ratio = 0.15
        n_train = len(X_train)
        val_size = max(1, int(n_train * val_ratio))

        X_tr = X_train[:-val_size]
        X_val = X_train[-val_size:]
        y_tr = y_train.iloc[:-val_size]
        y_val = y_train.iloc[-val_size:]

        classes = np.unique(y_tr)
        class_weights = compute_class_weight(
            class_weight="balanced",
            classes=classes,
            y=y_tr
        )
        class_weight_dict = {int(cls): float(w) for cls, w in zip(classes, class_weights)}

        cleanup_tf()

        model = build_rnn_attention_light(
            num_features=X_tr.shape[1],
            rnn_units=8,
            dropout_rate=0.2,
            learning_rate=1e-3,
            dense_units=8
        )

        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=3,
            min_delta=1e-3,
            restore_best_weights=True
        )

        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=25,
            batch_size=32,
            class_weight=class_weight_dict,
            callbacks=[early_stop],
            verbose=0
        )

        test_prob = fast_predict(model, X_test)
        y_pred = (test_prob >= 0.5).astype(int)   # 固定 threshold 0.5

        try:
            roc_auc = roc_auc_score(y_test, test_prob)
        except ValueError:
            roc_auc = np.nan

        fold_result = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc,
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_test, test_prob)
        }
        fold_metrics.append(fold_result)

        del model
        cleanup_tf()

        print(
            f"[{feature_name}] Fold {fold_idx} done | "
            f"ACC={fold_result['accuracy']:.4f} | "
            f"F1={fold_result['f1']:.4f} | "
            f"P={fold_result['precision']:.4f} | "
            f"R={fold_result['recall']:.4f}"
        )

    row = {
        "Model": "Light RNN + Attention",
        "Feature_Set": feature_name,
        "Config": "{rnn_units: 8, dropout: 0.2, lr: 1e-3, dense: 8, threshold: 0.5}",
        "Num_Features": len(selected_cols),
        "Num_Continuous": len([c for c in selected_cols if c in (semantic_cols + financial_cols)]),
        "Num_Lexical": len([c for c in selected_cols if c in lexical_cols]),
        "Accuracy_mean": np.nanmean([m["accuracy"] for m in fold_metrics]),
        "F1_mean": np.nanmean([m["f1"] for m in fold_metrics]),
        "ROC_AUC_mean": np.nanmean([m["roc_auc"] for m in fold_metrics]),
        "Precision_mean": np.nanmean([m["precision"] for m in fold_metrics]),
        "Recall_mean": np.nanmean([m["recall"] for m in fold_metrics]),
        "PR_AUC_mean": np.nanmean([m["average_precision"] for m in fold_metrics]),
        "Accuracy_std": np.nanstd([m["accuracy"] for m in fold_metrics], ddof=1),
        "F1_std": np.nanstd([m["f1"] for m in fold_metrics], ddof=1),
        "ROC_AUC_std": np.nanstd([m["roc_auc"] for m in fold_metrics], ddof=1),
        "Precision_std": np.nanstd([m["precision"] for m in fold_metrics], ddof=1),
        "Recall_std": np.nanstd([m["recall"] for m in fold_metrics], ddof=1),
        "PR_AUC_std": np.nanstd([m["average_precision"] for m in fold_metrics], ddof=1),
    }
    return row


# =========================================================
# N. 執行 M1~M6
# =========================================================
all_results = []

for feature_name, cols in feature_sets.items():
    result_row = evaluate_feature_set(
        df=df,
        y=y,
        groups=groups_all,
        feature_name=feature_name,
        selected_cols=cols
    )
    all_results.append(result_row)


# =========================================================
# O. 結果整理
# =========================================================
results_df = pd.DataFrame(all_results)
numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

print("\n===== Final Results =====")
print(results_df)

output_name = "chatgpt_light_rnn_attention_M1_M6_results.csv"
results_df.to_csv(output_name, index=False, encoding="utf-8-sig")

print(f"\nResults saved to: {output_name}")


Feature Set: M1: Semantic

[M1: Semantic] Fold 1
[M1: Semantic] Fold 1 done | ACC=0.6364 | F1=0.5455 | P=0.3750 | R=1.0000

[M1: Semantic] Fold 2
[M1: Semantic] Fold 2 done | ACC=0.9298 | F1=0.6667 | P=0.6667 | R=0.6667

[M1: Semantic] Fold 3
[M1: Semantic] Fold 3 done | ACC=0.9231 | F1=0.5714 | P=0.4000 | R=1.0000

[M1: Semantic] Fold 4
[M1: Semantic] Fold 4 done | ACC=0.9726 | F1=0.0000 | P=0.0000 | R=0.0000

[M1: Semantic] Fold 5
[M1: Semantic] Fold 5 done | ACC=0.8000 | F1=0.5517 | P=0.3810 | R=1.0000

Feature Set: M2: Lexical

[M2: Lexical] Fold 1
[M2: Lexical] Fold 1 done | ACC=0.2182 | F1=0.3582 | P=0.2182 | R=1.0000

[M2: Lexical] Fold 2
[M2: Lexical] Fold 2 done | ACC=0.9474 | F1=0.8000 | P=0.6667 | R=1.0000

[M2: Lexical] Fold 3
[M2: Lexical] Fold 3 done | ACC=0.9487 | F1=0.0000 | P=0.0000 | R=0.0000

[M2: Lexical] Fold 4
[M2: Lexical] Fold 4 done | ACC=0.8493 | F1=0.0000 | P=0.0000 | R=0.0000

[M2: Lexical] Fold 5
[M2: Lexical] Fold 5 done | ACC=0.7538 | F1=0.3333 | P=0.250